In [142]:
import threading
import queue
import numpy as np
import cv2
from ultralytics import YOLO
import cvzone
import pyttsx3
import speech_recognition as sr


In [143]:
engine = pyttsx3.init()
voices = engine.getProperty('voices')
engine.setProperty('rate', 150)
engine.setProperty('voice', voices[1].id)

In [144]:
speech_queue = queue.Queue()

In [145]:
def set_mode(new_mode):
    global mode
    mode = new_mode
    

In [146]:
def enQueue_text(text):
    speech_queue.put(text)

### Text to Speech Handling

In [147]:
def speak():
    while True:
        text = speech_queue.get()
        if text is None:
            break
        engine.say(text)
        engine.runAndWait()

### Speech to Text Handling

In [148]:
def listen_command():
    recogniser = sr.Recognizer()
    mic = sr.Microphone()
    while True:
        try:
            with mic as source:
                recogniser.adjust_for_ambient_noise(source)
                print('Listening...')
                audio = recogniser.listen(source)
                commands = recogniser.recognize_google(audio).lower()
                if 'normal' in commands:
                    enQueue_text('Normal mode started')
                    set_mode('normal')
                elif 'combine' in commands:
                    enQueue_text('combine mode started')
                    set_mode('combine')
                elif 'stop' in commands:
                    enQueue_text('Take care and goodbye')
                    set_mode('stop')
                    break
        except sr.UnknownValueError:
            print("Can't understand what you say")
        except sr.RequestError:
            print("Check your internet connection")
        except Exception as e:
            print(e)

In [149]:
! flac --version

flac 1.5.0


In [150]:
def calculate_elbow_angle(a,b,c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    angle_rad = np.arctan2(c[1]-b[1],c[0]-b[0]) - np.arctan2(a[1]-b[1],a[0]-b[0])
    angle_deg = np.abs((angle_rad*180)/np.pi)
    if angle_deg > 180:
        angle_deg = 360 - angle_deg
    return angle_deg

In [151]:
model = YOLO('yolo11n-pose.pt')

In [152]:
capFrame = cv2.VideoCapture(0)

In [153]:
mode = None
upper_threshold = 90
down_threshold = 150
left_arm_counter = 0
right_arm_counter = 0
right_arm_done = False
left_arm_done = False
combine = False
combine_counter = 0

In [154]:
speak_thread = threading.Thread(target=speak ,daemon=True)
speak_thread.start()
listen_thread = threading.Thread(target=listen_command ,daemon=True)
listen_thread.start()

### Main loop

In [155]:
while True:
    ret, frame = capFrame.read()
    if not ret:
        break
    frame = cv2.resize(frame,(1020,500))

    result = model.track(frame)

    if result[0].keypoints is not None:
        keypoints = result[0].keypoints.xy.cpu().numpy()

        for keypoint in keypoints:
            if len(keypoint) > 0:
                for i, point in enumerate(keypoint):
                    cx, cy = int(point[0]), int(point[1])
                    cv2.circle(frame,(cx,cy),5,(255,0,0),-1)
                    cvzone.putTextRect(frame,f'{i}',(cx,cy),1,1)

                if mode and len(keypoint) > 10:
                    left_shoulder = (int(keypoint[5][0]),int(keypoint[5][1]))
                    left_elbow = (int(keypoint[7][0]),int(keypoint[7][1]))
                    left_wrist = (int(keypoint[9][0]),int(keypoint[9][1]))

                    right_shoulder = (int(keypoint[6][0]),int(keypoint[6][1]))
                    right_elbow = (int(keypoint[8][0]),int(keypoint[8][1]))
                    right_wrist = (int(keypoint[10][0]),int(keypoint[10][1]))

                    left_arm_angle = calculate_elbow_angle(left_shoulder,left_elbow,left_wrist)
                    right_arm_angle = calculate_elbow_angle(right_shoulder,right_elbow,right_wrist)

                    cvzone.putTextRect(frame,f'Left Arm Angle: {int(left_arm_angle)}',(50, 50), 1, 1, colorR=(255, 0, 0))
                    cvzone.putTextRect(frame,f'Right Arm Angle: {int(right_arm_angle)}',(50, 50), 1, 1, colorR=(255, 0, 0))

                    if mode == 'normal':
                        if left_arm_angle < down_threshold and not left_arm_done:
                            left_arm_done = True
                        elif left_arm_angle > upper_threshold and left_arm_done:
                            left_arm_counter +=1
                            left_arm_done = False
                            enQueue_text(f'Left {left_arm_counter}')

                        if right_arm_angle < down_threshold and not right_arm_done:
                            right_arm_done = True
                        elif right_arm_angle > upper_threshold and right_arm_done:
                            right_arm_counter +=1
                            right_arm_done = False
                            enQueue_text(f'Right {right_arm_counter}')

                    elif mode == 'combine':
                        right_arm_counter = 0
                        left_arm_counter = 0
                        if right_arm_angle <= down_threshold and left_arm_angle <= down_threshold and not combine:
                            combine = True
                        elif right_arm_angle >= upper_threshold and left_arm_angle >= upper_threshold and combine:
                            combine_counter +=1
                            combine = False
                            enQueue_text(f'combine {combine_counter}')
    
    if mode == 'normal':
        cvzone.putTextRect(frame, f'Left hand counter: {int(left_arm_counter)}', (50, 110), 1, 1, colorR=(0, 0, 0))
        cvzone.putTextRect(frame, f'Right hand counter: {int(right_arm_counter)}', (50, 140), 1, 1, colorR=(0, 0, 0))
    elif mode == 'combine':
        cvzone.putTextRect(frame, f'Combine counter: {int(combine_counter)}', (50, 170), 1, 1, colorR=(0, 0, 0))

    # Display the frame
    cv2.imshow("RGB", frame)

    # Exit on 'Esc' key press
    key = cv2.waitKey(1)
    if mode=='stop':
        break

# Release resources
capFrame.release()
cv2.destroyAllWindows()

Can't understand what you say

Listening...
0: 320x640 1 person, 256.9ms
Speed: 5.0ms preprocess, 256.9ms inference, 4.9ms postprocess per image at shape (1, 3, 320, 640)

Listening...
0: 320x640 1 person, 192.1ms
Speed: 4.6ms preprocess, 192.1ms inference, 2.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 person, 195.8ms
Speed: 4.2ms preprocess, 195.8ms inference, 2.6ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 person, 216.2ms
Speed: 4.4ms preprocess, 216.2ms inference, 2.9ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 person, 185.7ms
Speed: 4.0ms preprocess, 185.7ms inference, 2.7ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 person, 172.1ms
Speed: 4.7ms preprocess, 172.1ms inference, 2.8ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 person, 217.9ms
Speed: 4.3ms preprocess, 217.9ms inference, 3.1ms postprocess per image at shape (1, 3, 320, 640)

0: 320x640 1 person, 186.9ms
Speed: 6.1ms prepr